In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# 1. 파일 경로 설정
# ============================================================
# 🌟 기존의 V4 학습에 썼던 카카오+LLM 텍스트 기반 Train 데이터
kakao_train_file = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/train_metadata.jsonl"

# 🌟 제미나이로 새롭게 뽑아낸 풍부한 텍스트 데이터 (파일명은 예시이므로 맞춰서 수정해주세요!)
gemini_full_file = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/metadata_gemini_finetune.jsonl"

# 🌟 두 개가 합쳐질 새로운 통합 Train 데이터
combined_train_file = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/train_multi_metadata.jsonl"

# ============================================================
# 2. 카카오 Train 데이터의 '이미지 파일명' 목록 추출
# (💡 Train/Valid/Test가 섞이는 걸 막기 위해, 카카오 Train에 있는 사진만 골라냅니다)
# ============================================================
train_filenames = set()
with open(kakao_train_file, "r", encoding="utf-8") as f:
    kakao_lines = f.readlines()
    for line in kakao_lines:
        import json
        train_filenames.add(json.loads(line)["file_name"])

# ============================================================
# 3. 제미나이 데이터에서 Train 사진에 해당하는 것만 쏙쏙 뽑기
# ============================================================
gemini_train_lines = []
with open(gemini_full_file, "r", encoding="utf-8") as f:
    for line in f:
        import json
        if json.loads(line)["file_name"] in train_filenames:
            gemini_train_lines.append(line)

# ============================================================
# 4. 파일 합치고 저장하기 (카카오 1450줄 + 제미나이 1450줄 = 2900줄)
# ============================================================
import random

combined_lines = kakao_lines + gemini_train_lines
random.seed(42)
random.shuffle(combined_lines) # 모델이 골고루 보게 섞어줍니다

with open(combined_train_file, "w", encoding="utf-8") as f:
    f.writelines(combined_lines)

print("✅ 다중 캡션(Multi-Captioning) 통합 데이터셋이 완성되었습니다!")
print(f" 📦 카카오 텍스트: {len(kakao_lines)}장")
print(f" 📦 제미나이 텍스트: {len(gemini_train_lines)}장")
print(f" 🔥 총 훈련 데이터: {len(combined_lines)}장 -> {combined_train_file}")

Mounted at /content/drive
✅ 다중 캡션(Multi-Captioning) 통합 데이터셋이 완성되었습니다!
 📦 카카오 텍스트: 1459장
 📦 제미나이 텍스트: 1439장
 🔥 총 훈련 데이터: 2898장 -> /content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/train_multi_metadata.jsonl


In [2]:
# ============================================================
# 0. 설치
# ============================================================
!pip install -q transformers datasets peft accelerate torchao --upgrade

# ============================================================
# 1. 임포트 & 드라이브 마운트
# ============================================================
import os, json, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from peft import get_peft_model, LoraConfig
from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# 2. 경로 설정 (🌟 Valid 데이터 경로 추가!)
# ============================================================
BASE_MODEL_ID = "Bingsu/clip-vit-large-patch14-ko"

TRAIN_JSONL = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/train_multi_metadata.jsonl"
VALID_JSONL = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/valid_metadata.jsonl"
IMAGE_DIR   = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong"

# V4: 조기종료 및 검증이 추가된 새로운 아웃풋 폴더
OUTPUT_DIR = "/content/drive/MyDrive/CV_Project/my_large_satellite_clip_v5"
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# ============================================================
# 3. 데이터셋 (기존과 동일)
# ============================================================
class SatelliteDataset(Dataset):
    def __init__(self, jsonl_path, image_dir, processor, split_name="Train"):
        self.processor = processor
        self.split_name = split_name

        self.image_index = {}
        for root, _, files in os.walk(image_dir):
            for fname in files:
                if fname.endswith((".jpg", ".png")):
                    self.image_index[fname] = os.path.join(root, fname)

        self.samples = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                fname = item["file_name"]
                if fname in self.image_index:
                    self.samples.append({
                        "image_path": self.image_index[fname],
                        "text": item["text"]
                    })
        print(f"[{self.split_name}] 데이터 로딩 완료: {len(self.samples)}장")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s["image_path"]).convert("RGB")
        inputs = self.processor(
            text=s["text"], images=image, return_tensors="pt",
            padding="max_length", truncation=True, max_length=77
        )
        return {k: v.squeeze(0) for k, v in inputs.items()}

# ============================================================
# 4. 모델 & LoRA 설정
# ============================================================
print("\n모델 로딩 중...")
processor = CLIPProcessor.from_pretrained(BASE_MODEL_ID)
model     = CLIPModel.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.float32)

lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model = model.cuda()

# ============================================================
# 5. DataLoader (🌟 Train과 Valid 두 개를 만듭니다)
# ============================================================
train_dataset = SatelliteDataset(TRAIN_JSONL, IMAGE_DIR, processor, split_name="Train")
valid_dataset = SatelliteDataset(VALID_JSONL, IMAGE_DIR, processor, split_name="Valid")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
# 6. 학습 설정 (넉넉한 에포크와 Early Stopping)
# ============================================================
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

MAX_EPOCHS = 30
PATIENCE   = 5  # 5번 연속으로 점수가 안 오르면 종료

best_val_loss = float('inf')
patience_counter = 0

def contrastive_loss(image_embeds, text_embeds, logit_scale):
    image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
    text_embeds  = text_embeds  / text_embeds.norm(dim=-1, keepdim=True)
    logits_per_image = logit_scale * image_embeds @ text_embeds.T
    logits_per_text  = logits_per_image.T
    labels = torch.arange(len(image_embeds), device=image_embeds.device)
    loss_i = torch.nn.functional.cross_entropy(logits_per_image, labels)
    loss_t = torch.nn.functional.cross_entropy(logits_per_text,  labels)
    return (loss_i + loss_t) / 2

# ============================================================
# 7. 본격적인 학습 루프 (Train + Valid)
# ============================================================
print("\n🚀 실전 파인튜닝 시작 (Early Stopping 장착)!")

for epoch in range(MAX_EPOCHS):
    # ------------------ TRAIN ------------------
    model.train()
    total_train_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {k: v.cuda() for k, v in batch.items()}
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(**batch)
            logit_scale = model.base_model.model.logit_scale.exp()
            loss = contrastive_loss(outputs.image_embeds, outputs.text_embeds, logit_scale)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # ------------------ VALID (검증 타임) ------------------
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.cuda() for k, v in batch.items()}
            with torch.amp.autocast('cuda'):
                outputs = model(**batch)
                logit_scale = model.base_model.model.logit_scale.exp()
                loss = contrastive_loss(outputs.image_embeds, outputs.text_embeds, logit_scale)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(valid_loader)

    print(f"[Epoch {epoch+1:02d}/{MAX_EPOCHS}] Train Loss: {avg_train_loss:.4f} | Valid Loss: {avg_val_loss:.4f}")

    # ------------------ BEST MODEL 저장 및 조기 종료 판단 ------------------
    if avg_val_loss < best_val_loss:
        print(f"  🌟 최고 성능 갱신! (Valid Loss: {best_val_loss:.4f} -> {avg_val_loss:.4f}) 모델을 저장합니다.")
        best_val_loss = avg_val_loss
        model.save_pretrained(BEST_MODEL_DIR)
        processor.save_pretrained(BEST_MODEL_DIR)
        patience_counter = 0 # 카운터 초기화
    else:
        patience_counter += 1
        print(f"  ⚠️ Valid Loss가 개선되지 않았습니다. (Patience: {patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:
        print(f"\n🚨 {PATIENCE}회 연속으로 Valid 점수가 오르지 않아 학습을 조기 종료합니다. (과적합 방지)")
        break

print(f"\n🎉 모든 학습 완료! 가장 똑똑한 모델이 저장된 곳: {BEST_MODEL_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

모델 로딩 중...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

[Train] 데이터 로딩 완료: 2898장
[Valid] 데이터 로딩 완료: 182장

🚀 실전 파인튜닝 시작 (Early Stopping 장착)!
[Epoch 01/30] Train Loss: 1.4334 | Valid Loss: 1.9031
  🌟 최고 성능 갱신! (Valid Loss: inf -> 1.9031) 모델을 저장합니다.
[Epoch 02/30] Train Loss: 1.1095 | Valid Loss: 1.8417
  🌟 최고 성능 갱신! (Valid Loss: 1.9031 -> 1.8417) 모델을 저장합니다.
[Epoch 03/30] Train Loss: 0.9735 | Valid Loss: 1.8168
  🌟 최고 성능 갱신! (Valid Loss: 1.8417 -> 1.8168) 모델을 저장합니다.
[Epoch 04/30] Train Loss: 0.9066 | Valid Loss: 1.9030
  ⚠️ Valid Loss가 개선되지 않았습니다. (Patience: 1/5)
[Epoch 05/30] Train Loss: 0.8299 | Valid Loss: 1.8499
  ⚠️ Valid Loss가 개선되지 않았습니다. (Patience: 2/5)
[Epoch 06/30] Train Loss: 0.7614 | Valid Loss: 1.9517
  ⚠️ Valid Loss가 개선되지 않았습니다. (Patience: 3/5)
[Epoch 07/30] Train Loss: 0.7166 | Valid Loss: 1.9900
  ⚠️ Valid Loss가 개선되지 않았습니다. (Patience: 4/5)
[Epoch 08/30] Train Loss: 0.6869 | Valid Loss: 1.9698
  ⚠️ Valid Loss가 개선되지 않았습니다. (Patience: 5/5)

🚨 5회 연속으로 Valid 점수가 오르지 않아 학습을 조기 종료합니다. (과적합 방지)

🎉 모든 학습 완료! 가장 똑똑한 모델이 저장된 곳: /content/dri

In [6]:
# LoRA 파인튜닝 모델 성능 평가 코드

import os
import json
import torch
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset
from transformers import CLIPProcessor, CLIPModel
from peft import PeftModel
import torch.nn.functional as F

# ============================================================
# 1. 경로 설정 (🌟 성동구 Test 데이터로 변경!)
# ============================================================
BASE_MODEL_NAME = "Bingsu/clip-vit-large-patch14-ko"

# 🌟 방금 학습이 끝난 V4 폴더의 epoch_5 경로
LORA_DIR = "/content/drive/MyDrive/CV_Project/my_large_satellite_clip_v5/best_model"

# 🌟 10%로 잘라둔 성동구 Test 데이터
TEST_JSONL = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/test_metadata.jsonl"
IMAGE_DIR  = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 2. 평가 데이터셋 로더
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, jsonl_path, image_dir):
        self.samples = []
        self.image_index = {}

        for root, _, files in os.walk(image_dir):
            for fname in files:
                if fname.endswith((".jpg", ".png")):
                    self.image_index[fname] = os.path.join(root, fname)

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                fname = item["file_name"]
                if fname in self.image_index:
                    self.samples.append({
                        "image_path": self.image_index[fname],
                        "text": item["text"]
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ============================================================
# 3. 평가 함수 (Retrieval R@1, R@5)
# ============================================================
@torch.no_grad()
def evaluate_retrieval(model, processor, dataset):
    print(" ⏳ 파인튜닝 모델(성동구 Test) 임베딩 추출 중...")
    image_embeds_all = []
    text_embeds_all = []

    for i in tqdm(range(len(dataset))):
        sample = dataset[i]
        image = Image.open(sample["image_path"]).convert("RGB")
        text = sample["text"]

        inputs = processor(text=[text], images=image, return_tensors="pt", padding=True).to(device)

        if hasattr(model, 'base_model'):
           outputs = model.base_model(**inputs)
        else:
           outputs = model(**inputs)

        image_embeds_all.append(outputs.image_embeds)
        text_embeds_all.append(outputs.text_embeds)

    image_embeds_all = F.normalize(torch.cat(image_embeds_all), dim=-1)
    text_embeds_all = F.normalize(torch.cat(text_embeds_all), dim=-1)

    sim_matrix = text_embeds_all @ image_embeds_all.T

    top1_correct = 0
    top5_correct = 0
    total_cosine_sim = 0.0
    num_samples = len(dataset)

    for i in range(num_samples):
        total_cosine_sim += sim_matrix[i, i].item()
        top5_indices = sim_matrix[i].topk(5).indices

        if top5_indices[0].item() == i: top1_correct += 1
        if i in top5_indices.tolist():  top5_correct += 1

    print(f"  ✔️ Image Retrieval R@1 : {(top1_correct / num_samples) * 100:.2f}%")
    print(f"  ✔️ Image Retrieval R@5 : {(top5_correct / num_samples) * 100:.2f}%")
    print(f"  ✔️ 텍스트-이미지 평균 유사도 : {total_cosine_sim / num_samples:.4f}")

# ============================================================
# 4. 평가 함수 (🌟 서술형 Rich-context Zero-shot 버전)
# ============================================================
@torch.no_grad()
def evaluate_zeroshot_rich(model, processor, dataset):
    print(" ⏳ 서술형(Rich) 프롬프트 기반 Zero-shot 평가 중...")

    # 🌟 핵심: 단순 키워드가 아닌 구체적 묘사로 보기(Prompt) 작성
    rich_prompts = [
        "수많은 나무와 산책로가 조성되어 녹색 픽셀이 뚜렷하게 보이는 넓은 녹지 및 공원 위성 사진",
        "회색 지붕의 건물들이 빽빽하게 밀집해 있고 넓은 도로나 주차장이 발달한 상업 및 업무 구역 위성 사진",
        "대형 학교 건물과 함께 모래나 우레탄으로 덮인 넓은 운동장이 뚜렷하게 식별되는 교육 환경 위성 사진",
        "다양한 형태의 저층 주택과 아파트 단지들이 좁은 골목길을 따라 복잡하게 모여있는 일반 주거 구역 위성 사진"
    ]

    correct = 0
    total = len(dataset)

    for i in tqdm(range(total)):
        sample = dataset[i]
        image = Image.open(sample["image_path"]).convert("RGB")
        actual_text = sample["text"]

        # 모델에게 4개의 긴 문장 중 이 사진과 가장 어울리는 설명을 고르라고 지시
        inputs = processor(text=rich_prompts, images=image, return_tensors="pt", padding=True, truncation=True).to(device)

        if hasattr(model, 'base_model'):
            outputs = model.base_model(**inputs)
        else:
            outputs = model(**inputs)

        logits_per_image = outputs.logits_per_image
        pred_idx = logits_per_image.argmax(dim=-1).item()

        keywords = {
            0: ["공원", "녹지", "자연", "서울숲", "하천"],
            1: ["음식점", "카페", "상가", "상권", "역세권", "시장"],
            2: ["학교", "학원", "교육", "초등학교", "대학교"],
            3: ["주거", "주택", "아파트", "빌라", "건물"]
        }

        if any(keyword in actual_text for keyword in keywords[pred_idx]):
            correct += 1

    print(f"✔️ Zero-shot Accuracy : {(correct / total) * 100:.2f}%")

# ============================================================
# 실행 메인
# ============================================================
if __name__ == "__main__":
    print(f"성동구 공식 Test 데이터셋 로딩 중: {TEST_JSONL}")
    test_dataset = EvalDataset(TEST_JSONL, IMAGE_DIR)

    if len(test_dataset) == 0:
        print("🚨 데이터셋을 찾을 수 없습니다. 경로를 확인하세요.")
    else:
        processor = CLIPProcessor.from_pretrained(BASE_MODEL_NAME)

        print("\n" + "="*50)
        print("Fine-Tuned/v5 모델 (성동구 Test) 평가 시작")
        print("="*50)

        base_model = CLIPModel.from_pretrained(BASE_MODEL_NAME)
        lora_model = PeftModel.from_pretrained(base_model, LORA_DIR).to(device)
        lora_model.eval()

        evaluate_retrieval(lora_model, processor, test_dataset)
        evaluate_zeroshot(lora_model, processor, test_dataset)

        print("\n✅ 성동구 공식 평가 완료!")

성동구 공식 Test 데이터셋 로딩 중: /content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/test_metadata.jsonl

Fine-Tuned/v5 모델 (성동구 Test) 평가 시작


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

 ⏳ 파인튜닝 모델(성동구 Test) 임베딩 추출 중...


100%|██████████| 183/183 [00:14<00:00, 12.38it/s]


  ✔️ Image Retrieval R@1 : 2.19%
  ✔️ Image Retrieval R@5 : 12.02%
  ✔️ 텍스트-이미지 평균 유사도 : 0.2097
 ⏳ Zero-shot 분류 점수 계산 중...


100%|██████████| 183/183 [00:15<00:00, 11.87it/s]

  ✔️ Zero-shot Accuracy : 77.05%

✅ 성동구 공식 평가 완료!


In [7]:
import os
import json
import torch
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

# ============================================================
# 1. 경로 설정 (🌟 LoRA 경로 없음, 순정 베이스 모델만 사용!)
# ============================================================
BASE_MODEL_NAME = "Bingsu/clip-vit-large-patch14-ko"

# 🌟 평가용 Test 데이터 (기존과 동일하게 유지)
TEST_JSONL = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/test_metadata.jsonl"
IMAGE_DIR  = "/content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 2. 평가 데이터셋 로더
# ============================================================
class EvalDataset(Dataset):
    def __init__(self, jsonl_path, image_dir):
        self.samples = []
        self.image_index = {}

        for root, _, files in os.walk(image_dir):
            for fname in files:
                if fname.endswith((".jpg", ".png")):
                    self.image_index[fname] = os.path.join(root, fname)

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                fname = item["file_name"]
                if fname in self.image_index:
                    self.samples.append({
                        "image_path": self.image_index[fname],
                        "text": item["text"]
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ============================================================
# 3. 평가 함수 (Retrieval R@1, R@5)
# ============================================================
@torch.no_grad()
def evaluate_retrieval(model, processor, dataset):
    print(" ⏳ [순정 Baseline] 임베딩 추출 중...")
    image_embeds_all = []
    text_embeds_all = []

    for i in tqdm(range(len(dataset))):
        sample = dataset[i]
        image = Image.open(sample["image_path"]).convert("RGB")
        text = sample["text"]

        inputs = processor(text=[text], images=image, return_tensors="pt", padding=True).to(device)
        outputs = model(**inputs)

        image_embeds_all.append(outputs.image_embeds)
        text_embeds_all.append(outputs.text_embeds)

    image_embeds_all = F.normalize(torch.cat(image_embeds_all), dim=-1)
    text_embeds_all = F.normalize(torch.cat(text_embeds_all), dim=-1)

    sim_matrix = text_embeds_all @ image_embeds_all.T

    top1_correct = 0
    top5_correct = 0
    total_cosine_sim = 0.0
    num_samples = len(dataset)

    for i in range(num_samples):
        total_cosine_sim += sim_matrix[i, i].item()
        top5_indices = sim_matrix[i].topk(5).indices

        if top5_indices[0].item() == i: top1_correct += 1
        if i in top5_indices.tolist():  top5_correct += 1

    print(f"  ✔️ Image Retrieval R@1 : {(top1_correct / num_samples) * 100:.2f}%")
    print(f"  ✔️ Image Retrieval R@5 : {(top5_correct / num_samples) * 100:.2f}%")
    print(f"  ✔️ 텍스트-이미지 평균 유사도 : {total_cosine_sim / num_samples:.4f}")

# ============================================================
# 4. 평가 함수 (🌟 서술형 Rich-context Zero-shot 버전)
# ============================================================
@torch.no_grad()
def evaluate_zeroshot_rich(model, processor, dataset):
    print(" ⏳ [순정 Baseline] 서술형(Rich) 프롬프트 기반 Zero-shot 평가 중...")

    rich_prompts = [
        "수많은 나무와 산책로가 조성되어 녹색 픽셀이 뚜렷하게 보이는 넓은 녹지 및 공원 위성 사진",
        "회색 지붕의 건물들이 빽빽하게 밀집해 있고 넓은 도로나 주차장이 발달한 상업 및 업무 구역 위성 사진",
        "대형 학교 건물과 함께 모래나 우레탄으로 덮인 넓은 운동장이 뚜렷하게 식별되는 교육 환경 위성 사진",
        "다양한 형태의 저층 주택과 아파트 단지들이 좁은 골목길을 따라 복잡하게 모여있는 일반 주거 구역 위성 사진"
    ]

    correct = 0
    total = len(dataset)

    for i in tqdm(range(total)):
        sample = dataset[i]
        image = Image.open(sample["image_path"]).convert("RGB")
        actual_text = sample["text"]

        inputs = processor(text=rich_prompts, images=image, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model(**inputs)

        logits_per_image = outputs.logits_per_image
        pred_idx = logits_per_image.argmax(dim=-1).item()

        keywords = {
            0: ["공원", "녹지", "자연", "서울숲", "하천"],
            1: ["음식점", "카페", "상가", "상권", "역세권", "시장"],
            2: ["학교", "학원", "교육", "초등학교", "대학교"],
            3: ["주거", "주택", "아파트", "빌라", "건물"]
        }

        if any(keyword in actual_text for keyword in keywords[pred_idx]):
            correct += 1

    print(f"  ✔️ Rich-context Zero-shot Accuracy : {(correct / total) * 100:.2f}%")

# ============================================================
# 실행 메인
# ============================================================
if __name__ == "__main__":
    print(f"성동구 공식 Test 데이터셋 로딩 중: {TEST_JSONL}")
    test_dataset = EvalDataset(TEST_JSONL, IMAGE_DIR)

    if len(test_dataset) == 0:
        print("🚨 데이터셋을 찾을 수 없습니다. 경로를 확인하세요.")
    else:
        processor = CLIPProcessor.from_pretrained(BASE_MODEL_NAME)

        print("\n" + "="*50)
        print("🚀 [대조군 실험] 순정 Baseline 모델 평가 시작 (LoRA 없음)")
        print("="*50)

        # 🌟 핵심: PeftModel을 씌우지 않고 순정 베이스 모델을 바로 디바이스로 올립니다.
        baseline_model = CLIPModel.from_pretrained(BASE_MODEL_NAME).to(device)
        baseline_model.eval()

        evaluate_retrieval(baseline_model, processor, test_dataset)
        evaluate_zeroshot_rich(baseline_model, processor, test_dataset)

        print("\n✅ Baseline 평가 완료! 이제 V4, V5 모델의 결과와 나란히 비교해 보세요.")

성동구 공식 Test 데이터셋 로딩 중: /content/drive/MyDrive/CV_Project/data/raw/tiles_seongdong/test_metadata.jsonl

🚀 [대조군 실험] 순정 Baseline 모델 평가 시작 (LoRA 없음)


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

 ⏳ [순정 Baseline] 임베딩 추출 중...


100%|██████████| 183/183 [00:14<00:00, 12.85it/s]


  ✔️ Image Retrieval R@1 : 0.00%
  ✔️ Image Retrieval R@5 : 5.46%
  ✔️ 텍스트-이미지 평균 유사도 : 0.2703
 ⏳ [순정 Baseline] 서술형(Rich) 프롬프트 기반 Zero-shot 평가 중...


100%|██████████| 183/183 [00:15<00:00, 11.77it/s]

  ✔️ Rich-context Zero-shot Accuracy : 79.78%

✅ Baseline 평가 완료! 이제 V4, V5 모델의 결과와 나란히 비교해 보세요.
